# 크레온 플러스 1분봉 수집

반도체 15개 종목의 1분봉을 크레온 플러스에서 최대한 과거까지 받는 노트북입니다. **윈도우 전용**이고, 위에서부터 순서대로 실행하면 됩니다.

**단계**
1. 실행 환경 점검 (32비트 Python, 관리자 권한, 크레온 연결)
2. 종목 코드 확인
3. 공통 요청 함수
4. 전 종목 수집: 수정주가/무수정주가 (약 10분)
5. 수집 결과 점검

**결과**: `creon_1min/snapshots/<실행일>/` 아래에 저장됩니다.
- `adjusted/<종목코드>.csv.gz`, `raw/<종목코드>.csv.gz`
- `manifest.jsonl`: 종목별 요청 횟수와 종료 사유
- `collection_report.txt`: 점검 결과

**다시 실행**: 같은 날 다시 실행하면 이미 받은 파일은 건너뛰고 나머지만 받습니다. 중간에 끊겨도 처음부터 다시 받지 않습니다.

**실행 전 준비**
- 크레온 플러스를 **관리자 권한**으로 실행해 로그인
- VS Code도 **관리자 권한**으로 실행. SSH 연결로 돌리면 크레온에 연결되지 않으므로 **윈도우 화면에서** 실행
- **32비트 Python 3.11** 환경에 아래 버전으로 설치. 32비트용 빌드가 있는 조합으로 고정한 것입니다.

```
pip install pywin32 "pandas==2.0.3" "numpy<2" "psutil<7.1.2" ipykernel
```

In [ ]:
import ctypes
import json
import platform
import struct
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import win32com.client

## 0. 설정

In [ ]:
# ============================================================
# 수집 대상
# ============================================================

STOCKS = {
    # 종목코드: (종목명, 분야)
    "005930": ("삼성전자", "메모리·파운드리"),
    "000660": ("SK하이닉스", "메모리"),
    "000990": ("DB하이텍", "파운드리"),
    "042700": ("한미반도체", "후공정 장비"),
    "039030": ("이오테크닉스", "레이저 장비"),
    "240810": ("원익IPS", "전공정 장비"),
    "036930": ("주성엔지니어링", "전공정 장비"),
    "403870": ("HPSP", "전공정 장비"),
    "089030": ("테크윙", "테스트 장비"),
    "058470": ("리노공업", "테스트 부품"),
    "095340": ("ISC", "테스트 부품"),
    "357780": ("솔브레인", "소재"),
    "005290": ("동진쎄미켐", "소재"),
    "064760": ("티씨케이", "소모품"),
    "067310": ("하나마이크론", "패키징·테스트"),
}

# ============================================================
# 요청 설정
# ============================================================

# StockChart 필드 번호 -> 컬럼명 (응답은 필드 번호 오름차순으로 온다)
FIELDS = {
    0: "date",
    1: "time",
    2: "open",
    3: "high",
    4: "low",
    5: "close",
    8: "volume",
    9: "value",
}

# 알려진 한계(약 18.5만 개)보다 넉넉하게 잡아서, 우리 값이 아니라 서버 쪽 한계에서 멈추게 한다
REQUEST_COUNT = 1_000_000

# 시세 조회 제한 구분값 (0: 주문, 1: 시세, 2: 실시간)
LIMIT_TYPE_QUOTE = 1

# KRX 정규장만 받는다. NXT는 2025-03 출범, KRX 장후 봉은 2026-09부터 확인되어 당분간 조회 범위에 남아 있으므로 필요하면 나중에 받는다
EXCHANGE = "K"
KRX_SESSION = "2"

PRICE_KINDS = {
    # 저장 폴더명: 수정주가 여부
    "adjusted": True,
    "raw": False,
}

# 이 시각 전에 실행하면 오늘 봉은 아직 완성되지 않았으므로 버린다 (수능일 폐장 16:30 기준)
SESSION_FINISHED_HHMM = 1640

# ============================================================
# 저장 경로
# ============================================================

DATA_DIR = Path("creon_1min")
PROBE_DIR = DATA_DIR / "probe"
SNAPSHOT_DIR = DATA_DIR / "snapshots" / datetime.now().strftime("%Y%m%d")
MANIFEST_PATH = SNAPSHOT_DIR / "manifest.jsonl"
REPORT_PATH = SNAPSHOT_DIR / "collection_report.txt"

## 0-1. 공통 유틸 (리포트, 매니페스트)

In [ ]:
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# 받은 시세 데이터가 git에 올라가지 않도록 폴더 전체를 제외
(DATA_DIR / ".gitignore").write_text("*\n")

REPORT_PATH.write_text("", encoding="utf-8")


def report(*args):
    # 화면 출력과 동시에 파일에도 남긴다
    line = " ".join(str(arg) for arg in args)

    print(line)

    with open(REPORT_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records

## 1. 실행 환경 점검

In [ ]:
bits = struct.calcsize("P") * 8
is_admin = bool(ctypes.windll.shell32.IsUserAnAdmin())

report("Python:", platform.python_version(), f"{bits}bit")
report("pandas:", pd.__version__)
report("관리자 권한:", is_admin)

# 크레온 COM 객체는 32비트 전용이라 64비트 Python에서는 객체 생성부터 실패한다
if bits != 32:
    raise RuntimeError("32비트 Python으로 실행해야 합니다.")

if not is_admin:
    report("경고: 관리자 권한이 아닙니다. 아래 연결에서 실패하면 VS Code를 관리자 권한으로 다시 실행하세요.")

cybos = win32com.client.Dispatch("CpUtil.CpCybos")

report("크레온 연결:", cybos.IsConnect)
report("시세 조회 잔여 횟수:", cybos.GetLimitRemainCount(LIMIT_TYPE_QUOTE))
report("제한 초기화까지 남은 시간:", cybos.LimitRequestRemainTime)

if cybos.IsConnect != 1:
    raise RuntimeError("크레온 플러스에 연결되어 있지 않습니다. 관리자 권한으로 실행해 로그인했는지 확인하세요.")

## 2. 종목 코드 확인

In [ ]:
code_mgr = win32com.client.Dispatch("CpUtil.CpCodeMgr")

mismatched = {}

for code, (name, sector) in STOCKS.items():
    creon_name = code_mgr.CodeToName(f"A{code}")

    if creon_name != name:
        mismatched[code] = (name, creon_name)

report(f"종목 {len(STOCKS)}개 중 이름 불일치 {len(mismatched)}개:", mismatched)

## 3. 공통 요청 함수

분봉은 기간 요청을 쓸 수 없어서(공식 문서) **개수 요청 + 연속 조회**로 받습니다. 응답은 최신 봉부터 옵니다.

개수 요청에서는 `Continue` 값이 끝까지 1로 남을 수 있어, 더 과거 데이터가 안 나오는 경우도 종료 조건에 넣었습니다. 어떤 조건으로 끝났는지는 `stop_reason`에 남습니다.

In [ ]:
def wait_for_quote_limit():
    # 시세 조회 제한에 걸리기 직전이면 제한이 풀릴 때까지 기다린다
    while cybos.GetLimitRemainCount(LIMIT_TYPE_QUOTE) <= 1:
        time.sleep(cybos.LimitRequestRemainTime / 1000 + 0.05)


def build_chart(code, adjusted, exchange="K", krx_session="1", count=REQUEST_COUNT):
    chart = win32com.client.Dispatch("CpSysDib.StockChart")

    chart.SetInputValue(0, f"A{code}")
    chart.SetInputValue(1, ord("2"))  # 개수 요청
    chart.SetInputValue(4, count)
    chart.SetInputValue(5, list(FIELDS))
    chart.SetInputValue(6, ord("m"))  # 분봉
    chart.SetInputValue(7, 1)  # 1분 주기
    chart.SetInputValue(9, ord("1" if adjusted else "0"))
    chart.SetInputValue(12, ord(exchange))  # A: 전체, K: KRX, N: NXT
    chart.SetInputValue(13, ord(krx_session))  # 1: 애프터마켓 포함, 2: 정규장만

    return chart


def request_page(chart):
    wait_for_quote_limit()

    chart.BlockRequest()

    status = chart.GetDibStatus()

    if status != 0:
        raise RuntimeError(f"요청 실패 (status={status}): {chart.GetDibMsg1()}")

    received = chart.GetHeaderValue(3)

    # GetDataValue의 첫 인자는 필드 번호가 아니라 요청한 필드 안에서의 순서
    rows = [
        [chart.GetDataValue(column, row) for column in range(len(FIELDS))]
        for row in range(received)
    ]

    return pd.DataFrame(rows, columns=list(FIELDS.values()))


def bar_key(df):
    return df["date"].astype("int64") * 10000 + df["time"].astype("int64")


def fetch_all_bars(code, adjusted, exchange="K", krx_session="1", count=REQUEST_COUNT):
    chart = build_chart(code, adjusted, exchange, krx_session, count)

    pages = []
    received_total = 0
    oldest_key = None

    while True:
        page = request_page(chart)

        if page.empty:
            stop_reason = "empty_page"
            break

        page_oldest_key = bar_key(page).min()

        if oldest_key is not None and page_oldest_key >= oldest_key:
            stop_reason = "no_older_data"
            break

        oldest_key = page_oldest_key
        pages.append(page)
        received_total += len(page)

        if len(pages) % 20 == 0:
            print(f"  요청 {len(pages)}회, 누적 {received_total:,}개, 가장 오래된 봉 {oldest_key}")

        if not chart.Continue:
            stop_reason = "continue_0"
            break

        if received_total >= count:
            stop_reason = "request_count_reached"
            break

    bars = pd.concat(pages, ignore_index=True) if pages else pd.DataFrame(columns=list(FIELDS.values()))
    bars = bars.drop_duplicates(["date", "time"]).sort_values(["date", "time"], ignore_index=True)

    info = {
        "requests": len(pages),
        "received": received_total,
        "stop_reason": stop_reason,
        "max_bars_per_request": max((len(page) for page in pages), default=0),
    }

    return bars, info


def summarize_bars(bars):
    if bars.empty:
        return {"rows": 0}

    bars_per_day = bars.groupby("date").size()
    value_unit = (bars["value"] / (bars["close"] * bars["volume"]).where(lambda s: s > 0)).median()

    return {
        "rows": len(bars),
        "first": int(bar_key(bars).min()),
        "last": int(bar_key(bars).max()),
        "trading_days": int(bars["date"].nunique()),
        "median_bars_per_day": float(bars_per_day.median()),
        "min_time": int(bars["time"].min()),
        "max_time": int(bars["time"].max()),
        "value_over_close_x_volume": round(float(value_unit), 6),
    }

## 4. 전 종목 수집

점검에서 확인한 내용을 반영했습니다.
- 조회 한계: 리노공업 기준 2024-10-11까지 192,535개. 서버가 `Continue=0`으로 끝을 알려줌
- 수정주가는 분봉에도 적용됨. 다만 수정주가는 가격과 거래량이 반올림되고 이후 권리 변동에 따라 값이 바뀌므로 **무수정주가도 함께 저장**
- 시세 조회 제한은 15초당 60회, 요청당 최대 2,499개

파일은 다 쓴 뒤에 이름을 바꾸므로, 중간에 끊겨도 불완전한 파일이 완료 파일로 남지 않습니다. 실패한 종목은 건너뛰고 끝에 목록을 보여주니, 원인을 해결하고 이 셀을 다시 실행하면 됩니다.

In [ ]:
def snapshot_path(kind, code):
    return SNAPSHOT_DIR / kind / f"{code}.csv.gz"


def drop_unfinished_session(bars):
    now = datetime.now()

    if now.hour * 100 + now.minute >= SESSION_FINISHED_HHMM:
        return bars

    return bars[bars["date"] != int(now.strftime("%Y%m%d"))]


def save_bars(bars, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_name(path.name + ".tmp")

    bars.to_csv(temp_path, index=False, compression="gzip")
    temp_path.replace(path)


failures = []

for code, (name, sector) in STOCKS.items():
    for kind, adjusted in PRICE_KINDS.items():
        path = snapshot_path(kind, code)

        if path.exists():
            print(f"건너뜀 {code} {name} {kind}")
            continue

        started_at = time.time()

        try:
            bars, info = fetch_all_bars(code, adjusted, EXCHANGE, KRX_SESSION)
        except RuntimeError as error:
            failures.append((code, name, kind, str(error)))
            report(f"실패 {code} {name} {kind}: {error}")
            continue

        bars = drop_unfinished_session(bars)

        if bars.empty:
            failures.append((code, name, kind, info["stop_reason"]))
            report(f"실패 {code} {name} {kind}: 받은 봉 없음 ({info['stop_reason']})")
            continue

        save_bars(bars, path)

        info.update(code=code, kind=kind, seconds=round(time.time() - started_at, 1))
        append_jsonl(MANIFEST_PATH, info)

        report(f"완료 {code} {name} {kind}: {len(bars):,}개, {bars['date'].min()}~{bars['date'].max()}, {info['stop_reason']}")

report(f"\n실패 {len(failures)}건:", failures)

## 5. 수집 결과 점검

저장된 파일을 다시 읽어 점검합니다. 수정/무수정 비율은 하루 중앙값으로 구간을 나눠 분할 같은 권리 변동을 보여줍니다.

In [ ]:
def adjustment_segments(adjusted_bars, raw_bars):
    merged = adjusted_bars.merge(raw_bars, on=["date", "time"], suffixes=("_adj", "_raw"))
    merged["ratio"] = merged["close_adj"] / merged["close_raw"]

    daily_ratio = merged.groupby("date")["ratio"].median().round(3)
    segment_id = (daily_ratio != daily_ratio.shift()).cumsum()

    segments = daily_ratio.groupby(segment_id).agg(lambda s: f"{s.index[0]}~{s.index[-1]}:{s.iloc[0]}")

    return " ".join(segments)


manifest = {(record["code"], record["kind"]): record for record in load_jsonl(MANIFEST_PATH)}
checks = []
segments_by_code = {}

for code, (name, sector) in STOCKS.items():
    if not all(snapshot_path(kind, code).exists() for kind in PRICE_KINDS):
        continue

    adjusted_bars = pd.read_csv(snapshot_path("adjusted", code))
    raw_bars = pd.read_csv(snapshot_path("raw", code))

    prices = raw_bars[["open", "high", "low", "close"]]
    summary = summarize_bars(raw_bars)

    checks.append({
        "code": code,
        "rows": summary["rows"],
        "first": summary["first"],
        "last": summary["last"],
        "days": summary["trading_days"],
        "median_bars": summary["median_bars_per_day"],
        "min_time": summary["min_time"],
        "max_time": summary["max_time"],
        "stop": manifest.get((code, "raw"), {}).get("stop_reason"),
        "requests": manifest.get((code, "raw"), {}).get("requests"),
        "adj_rows_equal": len(adjusted_bars) == len(raw_bars),
        "bad_price": int(((prices <= 0).any(axis=1)).sum()),
        "bad_ohlc": int(((prices["high"] < prices.max(axis=1)) | (prices["low"] > prices.min(axis=1))).sum()),
        "zero_volume": int((raw_bars["volume"] == 0).sum()),
        "max_adj_return": round(float(adjusted_bars["close"].pct_change().abs().max()), 3),
    })

    segments_by_code[code] = adjustment_segments(adjusted_bars, raw_bars)

check_table = pd.DataFrame(checks)

report(f"\n[종목별 점검] 완료 {len(checks)}/{len(STOCKS)}개")
report(check_table.to_string(index=False))

report("\n[수정/무수정 비율 구간]")

for code, segments in segments_by_code.items():
    report(code, STOCKS[code][0], segments)

점검 때 받은 애프터마켓 포함 데이터와 비교합니다. 정규장 옵션이 더 과거까지 가는지, 겹치는 봉의 값이 같은지 확인합니다.

In [ ]:
COMPARE_CODE = "058470"

probe_path = PROBE_DIR / f"{COMPARE_CODE}_raw.csv"

if probe_path.exists() and snapshot_path("raw", COMPARE_CODE).exists():
    probe_bars = pd.read_csv(probe_path)
    regular_bars = pd.read_csv(snapshot_path("raw", COMPARE_CODE))

    overlap = regular_bars.merge(probe_bars, on=["date", "time"], suffixes=("_regular", "_probe"))

    report(f"\n[정규장 옵션 vs 애프터마켓 포함] {COMPARE_CODE}")
    report("정규장 옵션:", len(regular_bars), "개, 시작", regular_bars["date"].min())
    report("애프터마켓 포함:", len(probe_bars), "개, 시작", probe_bars["date"].min())
    report("겹치는 봉:", len(overlap))
    report("종가 다른 봉:", int((overlap["close_regular"] != overlap["close_probe"]).sum()))
    report("거래량 다른 봉:", int((overlap["volume_regular"] != overlap["volume_probe"]).sum()))

report(f"\n리포트 저장 위치: {REPORT_PATH.resolve()}")